# Filament segmentation — U-Net training on Colab

Trains the runs named in `RUN_NAMES` (parameters cell; the whole experiment queue is the
`EXPERIMENTS` dict there, see `docs/PLAN.md`), writes TTA probability maps for the
held-out val split and the test set, and uploads each finished run to a **private** Kaggle
dataset `<owner>/filseg-<run>` so it can be pulled to the local machine for post-processing
tuning, ensembling and submission:

```bash
kaggle datasets download nguyenchicuong/filseg-<run> -p outputs/colab/<run> --unzip
```

Training state is written to Drive after every epoch, so a session that is cut off carries on
from the last epoch when the notebook is run again; a finished run is skipped.

**Before running**

1. Locally: `python scripts/package_colab.py`, then upload `build/colab/filseg-colab.zip` to
   `MyDrive/filseg/`. Upload it again whenever the code changes.
2. A Colab secret **`KAGGLE_API_TOKEN`** (🔑 in the left sidebar, notebook access on) holding the
   contents of `~/.kaggle/access_token` (`cat ~/.kaggle/access_token`), not a `kaggle.json`.
   The account must have joined the competition.
3. Runtime → Change runtime type → GPU (L4 or A100; a T4 works with `--bs 4` at 1024 px).

| GPU | ResNet34, 1024 px, bs 8, 30 epochs (rough estimate) |
|---|---|
| A100 | ~1 h |
| L4 | ~2 h |
| T4 | ~4 h (bs 4) |

EfficientNet-B4 / ConvNeXt-Tiny take roughly 1.5–2x as long; 60 epochs twice as long.
Several runs fit one session on an A100: list them in `RUN_NAMES`.

In [ ]:
# colab-only
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# colab-only
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# parameters
WORK = '/content/drive/MyDrive/filseg'          # bundle + per-epoch training state live here
BUNDLE = f'{WORK}/filseg-colab.zip'
CODE = '/content/filseg'                          # where the bundle is unpacked
OUT = '/content/out'                              # probability maps, before upload
OWNER = 'nguyenchicuong'                          # Kaggle account that receives the private datasets
COMPETITION = 'filament-segmentation-2026'

# The experiment queue (docs/PLAN.md). Each entry: arguments for scripts/train_unet.py.
# --split train keeps the fixed val split held out (needed to tune post-processing and to compare);
# --split all trains on every image for the final model (no val maps are written then).
# Batch sizes are for a 24 GB GPU (L4) or larger; on a T4 (16 GB) halve them.
EXPERIMENTS = {
    # E7  baseline at full working resolution
    'unet1024_r34':      '--res 1024 --bs 8 --epochs 30 --encoder resnet34',
    # E8  (#2) second output channel: the group map that decides which fragments form one filament
    'unet1024_r34_grp':  '--res 1024 --bs 8 --epochs 30 --encoder resnet34 --targets group',
    # E9  (#4) stronger encoders
    'unet1024_effb4':    '--res 1024 --bs 6 --epochs 30 --encoder efficientnet-b4',
    'unet1024_cnxt':     '--res 1024 --bs 6 --epochs 30 --encoder tu-convnext_tiny --lr 5e-4',
    # E10 (#4) longer training (768 was still improving at epoch 30)
    'unet1024_r34_e60':  '--res 1024 --bs 8 --epochs 60 --encoder resnet34',
    # E11 (#5) final model on every image: copy the winning recipe here once E7-E10 are compared
    'unet1024_final_full': '--res 1024 --bs 8 --epochs 30 --encoder resnet34 --split all',
}
# Which of them this session trains, in order (a finished + uploaded run is skipped, a run cut
# short resumes from its last epoch on Drive). Each run is uploaded as soon as it finishes, and
# scripts/watch_colab.py on the local machine tunes and submits it right away.
# The whole comparison queue, ~8 h on an A100 / ~16 h on an L4; E11 comes later with the winner.
RUN_NAMES = ['unet1024_r34', 'unet1024_r34_grp', 'unet1024_effb4', 'unet1024_cnxt', 'unet1024_r34_e60']
RUNS = [dict(name=n, args=EXPERIMENTS[n]) for n in RUN_NAMES]
WORKERS = 8
UPLOAD = True                                     # push each finished run to Kaggle
RELEASE_GPU = True                                # disconnect the runtime at the end

In [ ]:
# colab-only
!pip install -q -U kaggle segmentation-models-pytorch==0.5.0
import pathlib, subprocess
from google.colab import userdata
secret = userdata.get('KAGGLE_API_TOKEN').strip()
assert not secret.startswith('{'), ("KAGGLE_API_TOKEN holds a kaggle.json file; it must hold the contents of "
                                    "~/.kaggle/access_token (cat ~/.kaggle/access_token)")
token_path = pathlib.Path.home() / '.kaggle' / 'access_token'
token_path.parent.mkdir(exist_ok=True)
token_path.write_text(secret)
token_path.chmod(0o600)
check = subprocess.run(['kaggle', 'competitions', 'files', COMPETITION, '--page-size', '1'],
                       capture_output=True, text=True)
assert check.returncode == 0 and 'MAGFiLO' in check.stdout, f"Kaggle token check failed:\n{check.stdout}\n{check.stderr}"
print('Kaggle token OK')

In [ ]:
# colab-only
import os
assert os.path.isfile(BUNDLE), f"{BUNDLE} is missing: run scripts/package_colab.py locally and upload the zip"
!rm -rf "$CODE" && mkdir -p "$CODE" && unzip -q "$BUNDLE" -d "$CODE"
os.chdir(CODE)
missing = [f for f in ('scripts/train_unet.py', 'scripts/infer_probs.py', 'scripts/kaggle_upload.py',
                       'src/filseg/dl.py') if not os.path.exists(f)]
assert not missing, f"bundle is missing {missing}: it is an old zip, upload the current one"
print('code in', CODE)

## Data

The competition files (~720 MB) download in a minute or two and unpack to `data/` inside the code
directory, the layout the scripts expect.

In [ ]:
import os, pathlib, subprocess, zipfile
data = pathlib.Path(CODE) / 'data'
root = data / 'MAGFiLO_1.0_Kaggle_2026'
if not root.exists():
    data.mkdir(exist_ok=True)
    subprocess.run(['kaggle', 'competitions', 'download', COMPETITION, '-p', str(data), '-q'], check=True)
    for z in data.glob('*.zip'):
        zipfile.ZipFile(z).extractall(data)
        z.unlink()
n_train = len(list((root / 'train' / 'train_images').glob('*.jpeg')))
n_test = len(list((root / 'test' / 'test_images').glob('*.jpeg')))
assert (n_train, n_test) == (707, 180), f"unexpected data: {n_train} train / {n_test} test images"
print(f'{n_train} train / {n_test} test images')

## Train, predict, upload

Each run trains with `--resume` against its state on Drive, then writes 8x-TTA probability maps
(uint8, at the training resolution) for the val split and the test set, and uploads checkpoint +
maps to Kaggle. A run that fails is reported and the next one still runs.

In [ ]:
import json, shutil, subprocess, sys, time
from pathlib import Path

def sh(cmd):
    print('>>', ' '.join(map(str, cmd)), flush=True)
    return subprocess.run(list(map(str, cmd))).returncode

failed = []
for run in RUNS:
    name, args = run['name'], run['args'].split()
    models = Path(WORK) / 'runs'
    out = Path(OUT) / name
    done_marker = models / f'{name}.uploaded'
    if done_marker.exists():
        print(f'{name}: already trained and uploaded, skipping (delete {done_marker} to redo)', flush=True)
        continue
    t0 = time.time()
    code_ = sh([sys.executable, 'scripts/train_unet.py', *args, '--name', name, '--out-dir', models,
                '--workers', WORKERS, '--resume'])
    ckpt = models / f'{name}.pt'
    if code_ != 0 or not ckpt.exists():
        failed.append(name)
        print(f'{name}: TRAINING FAILED (exit {code_})', flush=True)
        continue
    print(f'{name}: trained in {(time.time() - t0) / 60:.0f} min', flush=True)
    shutil.rmtree(out, ignore_errors=True)
    out.mkdir(parents=True)
    for f in (f'{name}.pt', f'{name}.json'):
        shutil.copy(models / f, out / f)
    ok = sh([sys.executable, 'scripts/infer_probs.py', '--ckpt', ckpt, '--which', 'test', '--tta',
             '--out', out / 'probs_test']) == 0
    if '--split' not in args or args[args.index('--split') + 1] == 'train':
        ok &= sh([sys.executable, 'scripts/infer_probs.py', '--ckpt', ckpt, '--which', 'val', '--tta',
                  '--out', out / 'probs_val']) == 0
    if not ok:
        failed.append(name)
        continue
    if UPLOAD:
        slug = 'filseg-' + name.replace('_', '-')
        meta = json.loads((out / f'{name}.json').read_text())
        if sh([sys.executable, 'scripts/kaggle_upload.py', '--dir', out, '--owner', OWNER, '--slug', slug,
               '--message', f"best val pq {meta.get('best_val_pq')}"]) != 0:
            failed.append(name + ' (upload)')
        else:
            done_marker.write_text(slug)
print('failed:', failed or 'none')

## Logs

The val PQ of each run (from its training log) — paste this back into the chat.

In [ ]:
for run in RUNS:
    p = Path(WORK) / 'runs' / f"{run['name']}.json"
    print(run['name'], json.loads(p.read_text()).get('best_val_pq') if p.exists() else 'not finished')

## Release the GPU

In [ ]:
# colab-only
# Reached only when every cell above ran; after an error the runtime stays up for a look.
if RELEASE_GPU:
    from google.colab import runtime
    runtime.unassign()